# Lab 11 — Dashboard de risco de fraude

## Objetivo

Este laboratório transforma os indicadores da camada Gold em um dashboard HTML interativo para acompanhamento gerencial.

O painel possui:

- KPIs executivos;
- tendência temporal;
- composição do risco por segmento;
- tabela detalhada;
- tooltips interativos.

O arquivo gerado funciona de forma autônoma no navegador e não exige servidor ou privilégios administrativos.

In [1]:
from pathlib import Path
import webbrowser
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pasta_projeto = Path.cwd().resolve()

if not (pasta_projeto / "dados" / "serving").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "serving").exists():
            pasta_projeto = pasta_pai
            break

arquivo_segmento = (
    pasta_projeto
    / "dados"
    / "serving"
    / "fraud_risk_export.csv"
)

arquivo_diario = (
    pasta_projeto
    / "dados"
    / "serving"
    / "daily_metrics_export.csv"
)

pasta_dashboard = (
    pasta_projeto
    / "dia3_insights_bi"
    / "lab11_dashboard"
)

arquivo_dashboard = pasta_dashboard / "dashboard_fraude.html"

assert arquivo_segmento.exists(), "CSV por segmento não encontrado."
assert arquivo_diario.exists(), "CSV diário não encontrado."

print("Gold por segmento:", arquivo_segmento)
print("Gold diária:", arquivo_diario)
print("Dashboard:", arquivo_dashboard)

Gold por segmento: C:\BigData\bigdata-curso-gabriel\dados\serving\fraud_risk_export.csv
Gold diária: C:\BigData\bigdata-curso-gabriel\dados\serving\daily_metrics_export.csv
Dashboard: C:\BigData\bigdata-curso-gabriel\dia3_insights_bi\lab11_dashboard\dashboard_fraude.html


In [2]:
fraud_risk = pd.read_csv(arquivo_segmento)

daily_metrics = pd.read_csv(
    arquivo_diario,
    parse_dates=["transaction_date"]
)

daily_metrics = daily_metrics.sort_values(
    "transaction_date"
).reset_index(drop=True)

daily_metrics["media_movel_fraudes_7d"] = (
    daily_metrics["fraudes"]
    .rolling(window=7, min_periods=1)
    .mean()
)

validacao_dashboard = pd.DataFrame({
    "Indicador": [
        "Segmentos",
        "Dias com movimentação",
        "Transações",
        "Fraudes",
        "Valor em risco"
    ],
    "Resultado": [
        len(fraud_risk),
        len(daily_metrics),
        int(fraud_risk["total_transacoes"].sum()),
        int(fraud_risk["qtd_fraudes"].sum()),
        round(fraud_risk["valor_em_risco"].sum(), 2)
    ]
})

validacao_dashboard

,Indicador,Resultado
0,Segmentos,3.00
1,Dias com movimentação,672.00
2,Transações,100000.00
3,Fraudes,1833.00
4,Valor em risco,314587.93


In [3]:
total_transacoes = int(
    fraud_risk["total_transacoes"].sum()
)

total_fraudes = int(
    fraud_risk["qtd_fraudes"].sum()
)

taxa_fraude_geral = (
    100 * total_fraudes / total_transacoes
)

valor_em_risco = float(
    fraud_risk["valor_em_risco"].sum()
)

print("Transações:", total_transacoes)
print("Fraudes:", total_fraudes)
print("Taxa de fraude:", round(taxa_fraude_geral, 2), "%")
print("Valor em risco:", round(valor_em_risco, 2))

Transações: 100000
Fraudes: 1833
Taxa de fraude: 1.83 %
Valor em risco: 314587.93


In [4]:
fraud_risk = fraud_risk.sort_values(
    "taxa_fraude_pct",
    ascending=False
).reset_index(drop=True)

cores_segmentos = {
    "High-Risk": "#D64545",
    "Standard": "#F0A202",
    "Premium": "#1B998B"
}

cores_barras = [
    cores_segmentos.get(segmento, "#40798C")
    for segmento in fraud_risk["segment"]
]

fraud_risk[
    [
        "segment",
        "total_transacoes",
        "qtd_fraudes",
        "taxa_fraude_pct",
        "valor_em_risco"
    ]
]

,segment,total_transacoes,qtd_fraudes,taxa_fraude_pct,valor_em_risco
0,High-Risk,9155,705.0,7.70,126451.07
1,Standard,29689,655.0,2.21,106828.42
2,Premium,61156,473.0,0.77,81308.44


In [5]:
def formatar_numero_br(valor, casas=2):
    texto = f"{valor:,.{casas}f}"
    return (
        texto
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

In [6]:
fig = make_subplots(
    rows=3,
    cols=4,

    specs=[
        [
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"}
        ],
        [
            {"type": "xy", "colspan": 4},
            None,
            None,
            None
        ],
        [
            {"type": "xy", "colspan": 2},
            None,
            {"type": "table", "colspan": 2},
            None
        ]
    ],

    subplot_titles=[
        "",
        "",
        "",
        "",
        "Evolução diária das fraudes",
        "Taxa de fraude por segmento",
        "Detalhamento por segmento"
    ],

    row_heights=[0.18, 0.38, 0.44],
    vertical_spacing=0.10,
    horizontal_spacing=0.07
)

In [7]:
fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_transacoes,
        number={
            "valueformat": ",.0f",
            "font": {"size": 36, "color": "#17324D"}
        },
        title={"text": "Total de transações"}
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_fraudes,
        number={
            "valueformat": ",.0f",
            "font": {"size": 36, "color": "#D64545"}
        },
        title={"text": "Fraudes identificadas"}
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=taxa_fraude_geral,
        number={
            "valueformat": ".2f",
            "suffix": "%",
            "font": {"size": 36, "color": "#F0A202"}
        },
        title={"text": "Taxa geral de fraude"}
    ),
    row=1,
    col=3
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=valor_em_risco,
        number={
            "valueformat": ",.2f",
            "prefix": "R$ ",
            "font": {"size": 32, "color": "#6C4AB6"}
        },
        title={"text": "Valor em risco"}
    ),
    row=1,
    col=4
)

In [8]:
fig.add_trace(
    go.Scatter(
        x=daily_metrics["transaction_date"],
        y=daily_metrics["fraudes"],
        mode="lines",
        name="Fraudes diárias",
        line={
            "color": "#D8A1A1",
            "width": 1
        },
        hovertemplate=(
            "<b>%{x|%d/%m/%Y}</b><br>"
            "Fraudes: %{y:.0f}"
            "<extra></extra>"
        )
    ),
    row=2,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=daily_metrics["transaction_date"],
        y=daily_metrics["media_movel_fraudes_7d"],
        mode="lines",
        name="Média móvel de 7 dias",
        line={
            "color": "#D64545",
            "width": 3
        },
        hovertemplate=(
            "<b>%{x|%d/%m/%Y}</b><br>"
            "Média de 7 dias: %{y:.2f}"
            "<extra></extra>"
        )
    ),
    row=2,
    col=1
)

In [9]:
dados_hover_segmento = fraud_risk[
    [
        "total_transacoes",
        "qtd_fraudes",
        "valor_em_risco"
    ]
].to_numpy()

fig.add_trace(
    go.Bar(
        x=fraud_risk["segment"],
        y=fraud_risk["taxa_fraude_pct"],
        marker_color=cores_barras,
        customdata=dados_hover_segmento,
        text=[
            f"{valor:.2f}%"
            for valor in fraud_risk["taxa_fraude_pct"]
        ],
        textposition="outside",
        showlegend=False,
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Taxa de fraude: %{y:.2f}%<br>"
            "Transações: %{customdata[0]:,.0f}<br>"
            "Fraudes: %{customdata[1]:,.0f}<br>"
            "Valor em risco: R$ %{customdata[2]:,.2f}"
            "<extra></extra>"
        )
    ),
    row=3,
    col=1
)

In [10]:
cabecalhos_tabela = [
    "Segmento",
    "Transações",
    "Clientes",
    "Fraudes",
    "Taxa",
    "Valor em risco"
]

valores_tabela = [
    fraud_risk["segment"],
    [
        formatar_numero_br(valor, 0)
        for valor in fraud_risk["total_transacoes"]
    ],
    [
        formatar_numero_br(valor, 0)
        for valor in fraud_risk["total_clientes"]
    ],
    [
        formatar_numero_br(valor, 0)
        for valor in fraud_risk["qtd_fraudes"]
    ],
    [
        f"{formatar_numero_br(valor, 2)}%"
        for valor in fraud_risk["taxa_fraude_pct"]
    ],
    [
        f"R$ {formatar_numero_br(valor, 2)}"
        for valor in fraud_risk["valor_em_risco"]
    ]
]

fig.add_trace(
    go.Table(
        columnwidth=[100, 90, 80, 70, 70, 110],

        header={
            "values": [
                f"<b>{cabecalho}</b>"
                for cabecalho in cabecalhos_tabela
            ],
            "fill_color": "#17324D",
            "font": {
                "color": "white",
                "size": 12
            },
            "align": "center",
            "height": 30
        },

        cells={
            "values": valores_tabela,
            "fill_color": [
                ["#F4F7FA", "#FFFFFF", "#F4F7FA"]
            ],
            "font": {
                "color": "#243746",
                "size": 11
            },
            "align": [
                "left",
                "right",
                "right",
                "right",
                "right",
                "right"
            ],
            "height": 28
        }
    ),
    row=3,
    col=3
)

In [11]:
fig.update_layout(
    title={
        "text": (
            "<b>TechPay — Monitoramento de Risco de Fraude</b>"
            "<br><sup>Visão executiva das transações de 2023 a 2024</sup>"
        ),
        "x": 0.5,
        "xanchor": "center"
    },
    separators=",.",
    template="plotly_white",
    height=1050,
    margin={
        "l": 60,
        "r": 40,
        "t": 115,
        "b": 60
    },

    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#FFFFFF",

    font={
        "family": "Arial",
        "color": "#243746"
    },

    showlegend=True,

    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 0.46,
        "xanchor": "center",
        "x": 0.5
    },

    hoverlabel={
        "bgcolor": "white",
        "font_size": 12
    }
)

fig.update_xaxes(
    title_text="Data",
    showgrid=False,
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="Quantidade de fraudes",
    gridcolor="#E6ECF2",
    rangemode="tozero",
    row=2,
    col=1
)

fig.update_xaxes(
    title_text="Segmento",
    showgrid=False,
    row=3,
    col=1
)

fig.update_yaxes(
    title_text="Taxa de fraude (%)",
    gridcolor="#E6ECF2",
    rangemode="tozero",
    row=3,
    col=1
)

In [12]:
fig.show()

In [13]:
fig.write_html(
    str(arquivo_dashboard),
    include_plotlyjs=True,
    full_html=True,
    config={
        "displaylogo": False,
        "responsive": True,
        "toImageButtonOptions": {
            "format": "png",
            "filename": "dashboard_fraude_techpay",
            "scale": 2
        }
    }
)

print("Dashboard gerado com sucesso:")
print(arquivo_dashboard)
print(
    "Tamanho:",
    round(arquivo_dashboard.stat().st_size / (1024 ** 2), 2),
    "MB"
)

Dashboard gerado com sucesso:
C:\BigData\bigdata-curso-gabriel\dia3_insights_bi\lab11_dashboard\dashboard_fraude.html
Tamanho: 4.16 MB


In [14]:
webbrowser.open(arquivo_dashboard.resolve().as_uri())

print("Dashboard enviado ao navegador.")

Dashboard enviado ao navegador.


## O que cada bloco responde

- **Total de transações:** qual foi o volume de operações analisado?
- **Fraudes identificadas:** quantas ocorrências foram encontradas?
- **Taxa geral de fraude:** qual é a proporção de fraudes sobre o total?
- **Valor em risco:** qual é o valor financeiro das transações fraudulentas?
- **Tendência temporal:** como as fraudes evoluíram durante o período?
- **Composição por segmento:** qual segmento apresenta maior risco relativo?
- **Detalhamento:** quais são os números que sustentam a comparação entre os segmentos?

## Público e decisão apoiada

O dashboard foi concebido para a gerência de risco e prevenção a fraudes da TechPay. Seu objetivo é permitir o acompanhamento do nível geral de exposição, identificar segmentos prioritários e reconhecer alterações temporais que demandem investigação.

A equipe responsável pode utilizar o painel semanalmente para direcionar análises, revisar regras de monitoramento e priorizar segmentos com maior taxa ou valor financeiro em risco.

## Conclusão

O dashboard consolida os principais indicadores da camada Gold em um arquivo HTML interativo. Os KPIs apresentam o volume total, a quantidade e a taxa de fraudes e o valor financeiro em risco.

A série temporal permite acompanhar as ocorrências diárias e sua média móvel, reduzindo o efeito de oscilações pontuais. A comparação por segmento evidencia a maior exposição relativa do grupo High-Risk, enquanto a tabela detalhada disponibiliza os números utilizados na análise.

A solução utiliza Plotly e funciona sem servidor, banco de dados ativo ou conexão com a internet. Essa característica torna o dashboard adequado à rota sem privilégios administrativos adotada no curso.